### Setup and Load Data

This cell sets up the environment and loads the hard_examples.json file generated by the analysis notebook.

### Import and policy config

In [1]:
# --- Cell 1: imports & policy config ---
from pathlib import Path
import os, json, math, random
import numpy as np
import pandas as pd

# Repro
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Policy (FROZEN)
CANT_TELL_IN_S2 = False              # never allow label="cant_tell" in S2 few-shots
CANT_TELL_NEG_SHOTS = 2              # how many cant_tell exemplars to inject as negative few-shots (label="non")
CANT_TELL_RATIONALE = "Insufficient evidence for a concrete conspiracy claim; statements are ambiguous or hedged."

# Where to write outputs (use latest pipeline run)
DERIVED_ROOT = Path("data/derived")
LATEST_PTR = DERIVED_ROOT / "psycomark_latest.txt"
LATEST_DIR = Path(LATEST_PTR.read_text().strip())
LATEST_DIR


WindowsPath('C:/Users/panagiotis/Desktop/GitHub/PsyChoMark_Semeval/data/derived/psycomark_official_split_20251005_140323')

### Load data (S1/S2), folds, and any existing few-shots

In [2]:
# --- Cell 2: load data & folds ---
train_path      = LATEST_DIR / "train.jsonl"
dev_path        = LATEST_DIR / "dev.jsonl"
train_docclf    = LATEST_DIR / "train_docclf.jsonl"  # produced by data_pipeline.py
dev_docclf      = LATEST_DIR / "dev_docclf.jsonl"    # produced by data_pipeline.py
folds_path      = LATEST_DIR / "folds.jsonl"         # doc_id, dup_comp, fold (5-fold)

train_df   = pd.read_json(train_path, lines=True)
dev_df     = pd.read_json(dev_path, lines=True)

# For S2 few-shots, prefer the filtered (cant_tell removed) views if present
train_s2 = pd.read_json(train_docclf, lines=True) if train_docclf.exists() else train_df.copy()
dev_s2   = pd.read_json(dev_docclf, lines=True)   if dev_docclf.exists()   else dev_df.copy()

# Folds (optional use)
folds_df = pd.read_json(folds_path, lines=True) if folds_path.exists() else pd.DataFrame()

# Load existing few-shots if you want to preserve S1 from a prior run
existing_fs_path = LATEST_DIR / "best_fewshot_examples.json"
existing = json.loads(existing_fs_path.read_text()) if existing_fs_path.exists() else {"s1": [], "s2": []}

print("S2 label counts (train_docclf view):")
print(train_s2["doc_label"].value_counts(dropna=False))


S2 label counts (train_docclf view):
doc_label
non           1593
conspiracy    1161
cant_tell      606
Name: count, dtype: int64


### S2 helper: balanced pick with subreddit diversity (simple & robust)

In [3]:
# --- Cell 3: helpers for selection ---

def pick_balanced_by_label(df, label_col="doc_label", text_col="text", k_per_label=2, diversity_col="subreddit"):
    """
    Pick k_per_label examples from each of {'conspiracy','non'} with lightweight diversity by subreddit.
    """
    out = []
    for lab in ["conspiracy", "non"]:
        sub = df[df[label_col] == lab].copy()
        if sub.empty: 
            continue
        # ensure diversity: interleave by subreddit, then random within subreddit
        sub["_rand"] = np.random.default_rng(SEED).random(size=len(sub))
        order_cols = [diversity_col, "_rand"] if diversity_col in sub.columns else ["_rand"]
        sub = sub.sort_values(order_cols)
        out.extend(sub.head(k_per_label).to_dict(orient="records"))
    return out

def coerce_s2_item(r, label_override=None, rationale=None, tag=None):
    item = {
        "doc_id": r.get("doc_id"),
        "text": r.get("text", ""),
        "label": label_override if label_override else r.get("doc_label"),
        "rationale": rationale if rationale else "",
    }
    if tag:
        item["source_label"] = tag
    return item


### Build S2 few-shots under the frozen policy

In [4]:
# --- Cell 4: build S2 few-shots (policy enforced) ---

# Base pool for S2: only 'conspiracy' and 'non' (cant_tell excluded)
pool_main = pd.concat([train_s2, dev_s2], ignore_index=True)
pool_main = pool_main[pool_main["doc_label"].isin(["conspiracy","non"])].copy()

# Pick balanced few-shots from the main binary pool
S2_SHOTS_PER_CLASS = 2   # adjust if needed
s2_main_raw = pick_balanced_by_label(pool_main, k_per_label=S2_SHOTS_PER_CLASS)

# Optional: inject a small number of cant_tell as negative examples with an uncertainty rationale
s2_ct = []
if CANT_TELL_NEG_SHOTS > 0:
    all_raw = pd.concat([train_df[["doc_id","text","doc_label","subreddit"]],
                         dev_df[["doc_id","text","doc_label","subreddit"]]], ignore_index=True)
    ct_pool = all_raw[all_raw["doc_label"] == "cant_tell"].copy()
    if not ct_pool.empty:
        ct_pool["_rand"] = np.random.default_rng(SEED).random(size=len(ct_pool))
        order_cols = ["subreddit", "_rand"] if "subreddit" in ct_pool.columns else ["_rand"]
        ct_pool = ct_pool.sort_values(order_cols).head(CANT_TELL_NEG_SHOTS)
        for _, r in ct_pool.iterrows():
            s2_ct.append(coerce_s2_item(r, label_override="non", rationale=CANT_TELL_RATIONALE, tag="cant_tell"))

# Coerce main binary items into the output schema
s2_main = [coerce_s2_item(r) for r in s2_main_raw]
s2_fewshots = s2_main + s2_ct

# Safety: enforce the frozen policy
assert all(x["label"] in {"conspiracy","non"} for x in s2_fewshots), "Policy breach: S2 must be binary."
assert CANT_TELL_IN_S2 is False, "Policy breach: cant_tell cannot appear as S2 label."

len(s2_fewshots), sum(1 for x in s2_fewshots if x.get("source_label")=="cant_tell")


(6, 2)

### Keep existing S1 few-shots (if any) and save outputs

In [5]:
# --- Cell 5: merge with existing S1 few-shots (if present) and write files ---

fewshots = {
    "s1": existing.get("s1", []),  # keep whatever S1 you already curated
    "s2": s2_fewshots
}

# Write best_fewshot_examples.json next to other artifacts
out_fs_path = LATEST_DIR / "best_fewshot_examples.json"
with open(out_fs_path, "w", encoding="utf-8") as f:
    json.dump(fewshots, f, ensure_ascii=False, indent=2)

# Also write a small policy note for auditability
policy_meta = {
    "cant_tell": {
        "s2_training_excluded": True,
        "fewshot_negatives_added": int(sum(1 for ex in s2_fewshots if ex.get("source_label")=="cant_tell")),
        "rationale": CANT_TELL_RATIONALE
    },
    "seed": SEED
}
with open(LATEST_DIR / "fewshot_policy.json", "w", encoding="utf-8") as f:
    json.dump(policy_meta, f, indent=2)

out_fs_path, LATEST_DIR / "fewshot_policy.json"


(WindowsPath('C:/Users/panagiotis/Desktop/GitHub/PsyChoMark_Semeval/data/derived/psycomark_official_split_20251005_140323/best_fewshot_examples.json'),
 WindowsPath('C:/Users/panagiotis/Desktop/GitHub/PsyChoMark_Semeval/data/derived/psycomark_official_split_20251005_140323/fewshot_policy.json'))

### Quick sanity table

In [6]:
# --- Cell 6: sanity print ---
pd.DataFrame(s2_fewshots)[["label","source_label"]].value_counts(dropna=False)


label       source_label
conspiracy  NaN             2
non         cant_tell       2
            NaN             2
Name: count, dtype: int64

-----

### Load stats/priors

In [8]:
# --- Cell S1-A: load priors and overlap stats ---
import json
from pathlib import Path
import numpy as np
import pandas as pd

LATEST_DIR = Path((Path("data/derived") / "psycomark_latest.txt").read_text().strip())

priors_path  = LATEST_DIR / "length_position_priors.json"
pairs_path   = LATEST_DIR / "overlap_pair_stats.json"         # if present
pairs_ci_path= LATEST_DIR / "overlap_pair_stats_ci.json"      # with CIs (new)
boundary_ctx = LATEST_DIR / "boundary_context.json"

PRIORS = json.loads(priors_path.read_text()) if priors_path.exists() else {}
PAIR_STATS = json.loads(pairs_path.read_text()) if pairs_path.exists() else {}
PAIR_STATS_CI = json.loads(pairs_ci_path.read_text()) if pairs_ci_path.exists() else {}
BOUNDARY_CTX = json.loads(boundary_ctx.read_text()) if boundary_ctx.exists() else {}

ALLOWED = {"Actor","Action","Effect","Victim","Evidence"}


### Score candidates (prior-closeness + overlap bonus + diversity)

In [9]:
# --- Cell S1-B: candidate scoring ---
train_df = pd.read_json(LATEST_DIR / "train.jsonl", lines=True)

# Identify top ambiguous pairs to target
def top_pairs(pair_stats, key="iou@0.5", topk=2):
    items = [(p, d.get(key, 0.0)) for p, d in pair_stats.items()]
    items = sorted(items, key=lambda x: x[1], reverse=True)
    return [tuple(p.split("/")) for p,_ in items[:topk]]

TARGET_PAIRS = top_pairs(PAIR_STATS if PAIR_STATS else PAIR_STATS_CI, key="iou@0.5", topk=2)
TARGET_PAIRS = [tuple(sorted(p)) for p in TARGET_PAIRS]
TARGET_PAIRS

def beta_mode(alpha, beta):
    if alpha>1 and beta>1:
        return (alpha-1)/(alpha+beta-2)
    # fallback: mean
    return alpha/(alpha+beta) if (alpha>0 and beta>0) else 0.5

def prior_z_len(label, span_len):
    p = PRIORS.get(label, {}).get("length_lognorm", {"mu":0.0,"sigma":1.0})
    mu, sig = p["mu"], max(1e-6, p["sigma"])
    return abs((math.log(max(1,span_len)) - mu) / sig)

def prior_dist_pos(label, start_pos):
    sb = PRIORS.get(label, {}).get("start_beta", {"alpha":1.0,"beta":1.0})
    mode = beta_mode(sb["alpha"], sb["beta"])
    return abs(start_pos - mode)

def doc_has_target_overlap(markers):
    # Return +1 if the doc contains at least one span pair from TARGET_PAIRS with IoU >= 0.5
    def iou(a,b):
        s1,e1=a; s2,e2=b
        inter = max(0, min(e1,e2)-max(s1,s2))
        if inter<=0: return 0.0
        union = (e1-s1)+(e2-s2)-inter
        return inter/union if union>0 else 0.0
    spans = [(m["label"], m["start"], m["end"]) for m in (markers or []) if m.get("label") in ALLOWED]
    for i in range(len(spans)):
        l1,s1,e1 = spans[i]
        for j in range(i+1,len(spans)):
            l2,s2,e2 = spans[j]
            pair = tuple(sorted([l1,l2]))
            if pair in TARGET_PAIRS and iou((s1,e1),(s2,e2)) >= 0.5:
                return 1
    return 0

# Build a per-span candidate table
cand = []
for _, r in train_df.iterrows():
    t = r.get("text") or ""
    tlen = len(t)
    has_tpair = doc_has_target_overlap(r.get("markers"))
    for m in (r.get("markers") or []):
        L = m.get("label")
        if L not in ALLOWED: 
            continue
        s,e = int(m["start"]), int(m["end"])
        span_len = max(1, e-s)
        start_pos = s / max(1,tlen)
        score = 0.0
        # (1) closeness to priors (smaller is better) -> convert to positive score
        score += 1.5 * (1.0 - min(3.0, prior_z_len(L, span_len))/3.0)  # in [0,1]
        score += 1.5 * (1.0 - min(1.0, prior_dist_pos(L, start_pos)))   # in [0,1]
        # (2) overlap bonus for docs with target ambiguous pairs
        score += 1.0 * has_tpair
        # (3) light length cap to avoid giant excerpts
        score += 0.25 * (span_len <= PRIORS.get(L,{}).get("q90_len", 999))
        cand.append({
            "doc_id": r["doc_id"],
            "subreddit": r.get("subreddit","NA"),
            "label": L,
            "start": s, "end": e,
            "score": float(score),
            "text": t
        })
cands = pd.DataFrame(cand)
len(cands)


17328

### Pick S1 few-shots (1 per label near prior + small quota of outliers)

In [10]:
# --- Cell S1-C: select final S1 few-shots ---
S1_PER_LABEL = 2           # 2 exemplars per label
OUTLIER_QUOTA = 1          # per label; intentionally weird to teach boundaries
SUBREDDIT_DIVERSITY = True

def pick_for_label(df_lab, k=2, outlier_k=1):
    # take top by score, but also include a few outliers (largest z_len or farthest pos)
    topk = df_lab.sort_values("score", ascending=False)
    chosen = []
    if SUBREDDIT_DIVERSITY and "subreddit" in df_lab.columns:
        seen = set()
        for _, row in topk.iterrows():
            if row["subreddit"] in seen: 
                continue
            chosen.append(row)
            seen.add(row["subreddit"])
            if len(chosen) >= k: 
                break
    if len(chosen) < k:
        chosen = topk.head(k).to_dict(orient="records")
    else:
        chosen = [r._asdict() if hasattr(r, "_asdict") else dict(r) for r in chosen]

    # add outliers
    df_lab = df_lab.copy()
    df_lab["z_len"] = df_lab.apply(lambda r: prior_z_len(r["label"], r["end"]-r["start"]), axis=1)
    df_lab["pos_dist"] = df_lab.apply(lambda r: prior_dist_pos(r["label"], r["start"]/max(1,len(r["text"]))), axis=1)
    outliers = df_lab.sort_values(["z_len","pos_dist"], ascending=False).head(outlier_k)
    chosen += outliers.to_dict(orient="records")
    return chosen[:k+outlier_k]

s1_examples = []
for L in ALLOWED:
    pick_from = cands[cands["label"]==L]
    if pick_from.empty:
        continue
    picks = pick_for_label(pick_from, k=S1_PER_LABEL, outlier_k=OUTLIER_QUOTA)
    for p in picks:
        s,e = int(p["start"]), int(p["end"])
        # keep a compact window around span to keep token budget small
        pad = 120
        t = p["text"]
        left = max(0, s-pad)
        right = min(len(t), e+pad)
        snippet = (t[left:s] + t[s:e] + t[e:right]).strip()
        # normalize to the snippet's coordinate space
        new_start = len(t[left:s])
        new_end   = new_start + (e-s)
        s1_examples.append({
            "doc_id": p["doc_id"],
            "text": snippet,
            "spans": [{"label": p["label"], "start": new_start, "end": new_end}],
            "meta": {
                "source_window": [left, right],
                "reason": "prior_closeness+ambiguous_pair_bonus"
            }
        })

# Merge back into best_fewshot_examples.json (keep S2 from earlier)
out_fs_path = LATEST_DIR / "best_fewshot_examples.json"
if out_fs_path.exists():
    best = json.loads(out_fs_path.read_text())
else:
    best = {"s1": [], "s2": []}

best["s1"] = s1_examples
with open(out_fs_path, "w", encoding="utf-8") as f:
    json.dump(best, f, ensure_ascii=False, indent=2)

len(best["s1"])


15

-----------------------------

In [1]:
# CELL 1: Setup and Load Data
import json
import pandas as pd
from pathlib import Path

# --- CONFIGURATION ---
# Automatically find the latest pipeline run via the pointer file
OUTPUT_ROOT = Path("./data/derived")
latest_run_ptr = OUTPUT_ROOT / "psycomark_latest.txt"

if not latest_run_ptr.exists():
    raise FileNotFoundError(f"Could not find pointer file at '{latest_run_ptr}'. Please run data_pipeline.py first.")

PIPELINE_OUTPUT_DIR = Path(latest_run_ptr.read_text().strip())
print(f"--- Loading data from latest pipeline run: {PIPELINE_OUTPUT_DIR.name} ---")

# --- Load the 'hard_examples.json' and the full dev set for context ---
hard_examples_path = PIPELINE_OUTPUT_DIR / "hard_examples.json"
if not hard_examples_path.exists():
    raise FileNotFoundError(f"File '{hard_examples_path}' not found. Please run the analysis notebook first.")

with open(hard_examples_path, 'r', encoding='utf-8') as f:
    hard_examples = json.load(f)

# Load the full dataset to fetch marker and label info
train_df = pd.read_json(PIPELINE_OUTPUT_DIR / "train.jsonl", lines=True)
dev_df = pd.read_json(PIPELINE_OUTPUT_DIR / "dev.jsonl", lines=True)
df_all = pd.concat([train_df, dev_df], ignore_index=True).set_index('doc_id')

# Convert to a DataFrame for easier processing
df_hard = pd.DataFrame(hard_examples).set_index('doc_id')
df_hard = df_hard.join(df_all[['markers', 'doc_label']])
df_hard = df_hard[df_hard["doc_label"].isin(["conspiracy","non"])]

print(f"Loaded {len(df_hard)} candidate 'hard' examples.")
df_hard.head()

--- Loading data from latest pipeline run: psycomark_official_split_20250928_232947 ---
Loaded 869 candidate 'hard' examples.


,reasons,text,markers,doc_label
doc_id,,,,
t1_csthti6,[High Action/Effect IoU (1.00)],Saw this in /r/SandersForPresident and they se...,"[{'label': 'Evidence', 'start': 12, 'end': 34,...",non
t1_dp0t655,[High Action/Effect IoU (1.00)],Since my last post sparked a heated discussi...,"[{'label': 'Action', 'start': 31, 'end': 49, '...",non
t1_dt9elm2,"[High Action/Effect IoU (1.00), High Subreddit...",Wasserman-Shultz clearly stole the election in...,"[{'label': 'Actor', 'start': 0, 'end': 16, 'te...",conspiracy
t1_e1pv6xg,"[High Action/Effect IoU (1.00), High Subreddit...","Amy Manson, actress in the upcoming Doom movie...","[{'label': 'Actor', 'start': 0, 'end': 11, 'te...",conspiracy
t1_eac1hkd,"[High Action/Effect IoU (1.00), High Subreddit...",how many more Russians are going to get poison...,"[{'label': 'Victim', 'start': 14, 'end': 22, '...",conspiracy


### Define a Scoring Strategy to Rank Examples

Here, we define a function to score each example. The score will be based on how well an example fulfills our desired criteria: demonstrating ambiguity, covering different labels, and having a reasonable length.

In [2]:
# CELL 2: Define a Scoring Strategy to Rank Examples

def score_example(row):
    """
    Scores a candidate few-shot example based on a set of heuristics.
    Higher score is better.
    """
    score = 0
    reasons = row['reasons']
    
    # --- Criterion 1: Ambiguity and Complexity ---
    # High score for examples that are hard for multiple reasons
    score += len(reasons) * 10
    
    # Specific bonus for high Action/Effect or Actor/Victim overlap
    if any("High Action/Effect IoU" in r for r in reasons):
        score += 15
    if any("High Actor/Victim IoU" in r for r in reasons): # We'll need to add this to the hard examples script later
        score += 15
    
    # --- Criterion 2: Label Coverage ---
    # Give points for having a clear conspiracy or non-conspiracy label
    if row['doc_label'] == 'conspiracy':
        score += 10
    elif row['doc_label'] == 'non':
        score += 10
    # No points for 'cant_tell' as it's less instructive
    
    # --- Criterion 3: Marker Density ---
    # More markers are generally more instructive, up to a point
    num_markers = len(row['markers']) if isinstance(row['markers'], list) else 0
    score += min(num_markers, 5) * 2 # Add up to 10 points for marker density
    
    # Bonus for having a diverse set of marker labels
    if num_markers > 0:
        unique_labels = {m['label'] for m in row['markers']}
        score += len(unique_labels) * 3
        
    # --- Criterion 4: Text Length ---
    # Penalize very short or very long examples. Sweet spot is 200-600 chars.
    text_len = len(row['text'])
    if 200 < text_len < 600:
        score += 5
    elif text_len < 160 or text_len > 800:
        score -= 10
        
    return score

# Apply the scoring function
df_hard['score'] = df_hard.apply(score_example, axis=1)

# Display the top-scoring examples
df_hard_sorted = df_hard.sort_values('score', ascending=False)
print("Top 10 highest-scoring few-shot candidates:")
display(df_hard_sorted[['reasons', 'doc_label', 'score', 'text']].head(10))

Top 10 highest-scoring few-shot candidates:


,reasons,doc_label,score,text
doc_id,,,,
t1_gb06koo,"[High Action/Effect IoU (1.00), High Subreddit...",conspiracy,75,Your tax exempt cult is a poison on these land...
t1_dt9elm2,"[High Action/Effect IoU (1.00), High Subreddit...",conspiracy,75,Wasserman-Shultz clearly stole the election in...
t1_e1pv6xg,"[High Action/Effect IoU (1.00), High Subreddit...",conspiracy,75,"Amy Manson, actress in the upcoming Doom movie..."
t1_f76erur,"[High Action/Effect IoU (1.00), High Subreddit...",conspiracy,75,Perhaps Shokin was not a great prosecutor gene...
t1_erkidj3,"[High Action/Effect IoU (0.78), High Subreddit...",conspiracy,75,I think this ironic comment shows that the imm...
t1_eac1hkd,"[High Action/Effect IoU (1.00), High Subreddit...",conspiracy,75,how many more Russians are going to get poison...
t1_f7bzvwr,"[High Action/Effect IoU (1.00), High Subreddit...",conspiracy,75,"World War One killed millions of people, destr..."
t1_elcq7w6,"[High Action/Effect IoU (1.00), High Subreddit...",conspiracy,72,Israel's backers will shamelessly exploit any ...
t1_f3otibj,"[High Action/Effect IoU (1.00), High Subreddit...",non,72,I try not clog up this forum with stuff off r/...


### Select the Best N Examples and Export

This final cell selects the best examples based on our scoring. It ensures we have a balanced set (e.g., at least one 'conspiracy' and one 'non') and then exports them to a clean JSON file.

In [3]:
# ===== Selection for S1 and S2 =====
from collections import defaultdict
import json

N_S1_PER_LABEL = 2          # ~10 total for five labels
N_S2_PER_CLASS = 8          # 16 total; tune 6–10 if tokens are tight
MAX_PER_SUBREDDIT = 2

# Build a quick lookup to get full docs by id (you already loaded df_all)
def doc_by_id(did):
    return df_all.loc[did].to_dict()

def markers_compact(markers, max_per_label=2):
    # Keep only label/start/end; at most max_per_label per label to limit prompt length
    out = []
    per_label = defaultdict(int)
    for m in markers or []:
        lbl = m.get("label")
        if per_label[lbl] >= max_per_label: 
            continue
        if {"label","start","end"} <= m.keys():
            out.append({"label": m["label"], "start": int(m["start"]), "end": int(m["end"])})
            per_label[lbl] += 1
    return out

# ------------- S1: spans -------------
# Strategy: take best 'clear' per label + inject 2–3 high IoU Action/Effect
# Use your df_hard_sorted as pool; fallback to df_all if needed.
s1_examples = []
picked_s1_per_label = defaultdict(int)
seen_s1_docs = set()
seen_sr = defaultdict(int)

def try_add_s1(doc_id, prefer_labels=None):
    global s1_examples
    row = df_all.loc[doc_id]
    text = row["text"]
    markers = markers_compact(row.get("markers", []), max_per_label=1)
    if not markers: 
        return False
    # If focusing on a particular label, filter to that label
    if prefer_labels:
        marks = [m for m in markers if m["label"] in prefer_labels]
        if not marks: 
            return False
        markers = marks[:1]
    lbl = markers[0]["label"]
    sr = row.get("subreddit","")
    if picked_s1_per_label[lbl] >= N_S1_PER_LABEL: 
        return False
    if seen_sr[sr] >= MAX_PER_SUBREDDIT:
        return False
    s1_examples.append({"text": text, "markers": markers})
    picked_s1_per_label[lbl] += 1
    seen_sr[sr] += 1
    seen_s1_docs.add(doc_id)
    return True

# Pass 1: clear exemplars per label from the high-score pool
for doc_id, r in df_hard_sorted.iterrows():
    # Prefer docs whose markers include any needed label
    if all(picked_s1_per_label[l] >= N_S1_PER_LABEL for l in ["Actor","Action","Effect","Victim","Evidence"]):
        break
    mks = r["markers"] if isinstance(r["markers"], list) else []
    labels = {m["label"] for m in mks}
    for target in ["Actor","Action","Effect","Victim","Evidence"]:
        if target in labels and picked_s1_per_label[target] < N_S1_PER_LABEL:
            try_add_s1(doc_id, prefer_labels=[target])

# Pass 2: ensure we include 2–3 hard Action↔Effect overlaps specifically
hard_ae = df_hard_sorted[[any("Action/Effect" in rs for rs in r) for r in df_hard_sorted["reasons"]]]
ae_added = 0
for doc_id, r in hard_ae.head(6).iterrows():
    if ae_added >= 3:
        break
    if doc_id in seen_s1_docs: 
        continue
    if try_add_s1(doc_id, prefer_labels=["Action","Effect"]):
        ae_added += 1

# ------------- S2: docs -------------
# Balance conspiracy/non; exclude cant_tell; subreddit diversity
def pick_s2(class_label, k):
    picked, seen_sr2 = [], defaultdict(int)
    pool = df_hard_sorted[df_hard_sorted["doc_label"]==class_label]
    for doc_id, r in pool.iterrows():
        row = df_all.loc[doc_id]
        txt = row["text"]
        sr = row.get("subreddit","")
        if not (200 <= len(txt) <= 1200): 
            continue
        if seen_sr2[sr] >= MAX_PER_SUBREDDIT:
            continue
        picked.append({
            "text": txt, 
            "doc_label": class_label, 
            "rationale": ("Mentions multiple markers and alleges a coordinated, malicious plot."
                          if class_label=="conspiracy" 
                          else "Critical or political text without alleging a secret, coordinated, malicious plot.")
        })
        seen_sr2[sr] += 1
        if len(picked) >= k: 
            break
    return picked

s2_examples = []
s2_examples += pick_s2("conspiracy", N_S2_PER_CLASS)
s2_examples += pick_s2("non", N_S2_PER_CLASS)

# ------------- Save -------------
out_path = PIPELINE_OUTPUT_DIR / "best_fewshot_examples.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump({"s1": s1_examples, "s2": s2_examples}, f, ensure_ascii=False, indent=2)

print(f"Saved few-shots to: {out_path}")
print(f"S1 count: {len(s1_examples)} | per-label: {dict(picked_s1_per_label)}")
print(f"S2 count: {len(s2_examples)} (consp={N_S2_PER_CLASS}, non={N_S2_PER_CLASS})")


Saved few-shots to: C:\Users\panagiotis\Desktop\GitHub\PsyChoMark_Semeval\data\derived\psycomark_official_split_20250928_232947\best_fewshot_examples.json
S1 count: 10 | per-label: {'Actor': 2, 'Action': 2, 'Effect': 2, 'Victim': 2, 'Evidence': 2}
S2 count: 16 (consp=8, non=8)
